# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Storage Solutions (Neo4j)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [ ]:
!pip install graphframes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 KB 291.6 kB/s eta 0:00:0000:0100:01


In [ ]:
from spark_utils import SparkUtils
neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils("Example EDA", "spark://spark-master:7077", spark_packages=neo4j_connector)
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5cbbb2d0-f507-4502-9d4e-23a1d7fd4657;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv

# Create GraphFrames

In [3]:
from graphframes import GraphFrame

vertices = su.spark.createDataFrame([
  ("a", "Alice"),
  ("b", "Bob"),
  ("c", "Carol")
], ["id", "name"])

edges = su.spark.createDataFrame([
  ("a","b","follows"),
  ("b","c","follows"),
  ("c","a","follows")
], ["src","dst","relationship"])

g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

+---+-----+
| id| name|
+---+-----+
|  a|Alice|
|  b|  Bob|
|  c|Carol|
+---+-----+

+---+---+------------+
|src|dst|relationship|
+---+---+------------+
|  a|  b|     follows|
|  b|  c|     follows|
|  c|  a|     follows|
+---+---+------------+



# Core Graph Algorithms
## PageRank

In [28]:
results = g.pageRank(resetProbability=0.15, maxIter=10)
results.vertices.printSchema()
# Ranked vertices (descending by importance)
results.vertices \
  .select("id", "name", "pagerank") \
  .orderBy("pagerank", ascending=False) \
  .show()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- pagerank: double (nullable = true)

+---+-----+--------+
| id| name|pagerank|
+---+-----+--------+
|  a|Alice|     1.0|
|  c|Carol|     1.0|
|  b|  Bob|     1.0|
+---+-----+--------+



# Label Propagation

In [29]:
lpa = g.labelPropagation(maxIter=5)
lpa.show()

+---+-----+-----+
| id| name|label|
+---+-----+-----+
|  a|Alice|    0|
|  b|  Bob|    0|
|  c|Carol|    0|
+---+-----+-----+



# Triangle counting

In [6]:
triangle_count = g.triangleCount()
triangle_count.show()

+-----+---+-----+
|count| id| name|
+-----+---+-----+
|    1|  a|Alice|
|    1|  b|  Bob|
|    1|  c|Carol|
+-----+---+-----+



# Degrees Distribution

## InDregree

In [7]:
in_deg = g.inDegrees.join(vertices, "id")
in_deg.show()

+---+--------+-----+
| id|inDegree| name|
+---+--------+-----+
|  b|       1|  Bob|
|  a|       1|Alice|
|  c|       1|Carol|
+---+--------+-----+



## OutDegree

In [8]:
out_deg = g.outDegrees.join(vertices, "id")
out_deg.show()

+---+---------+-----+
| id|outDegree| name|
+---+---------+-----+
|  a|        1|Alice|
|  b|        1|  Bob|
|  c|        1|Carol|
+---+---------+-----+



# Write data to a Neo4j Graph

## Neo4j setup
### Install Neo4j with Docker

Go to **spark** directory and run:

```
docker run \
    -d --restart always \
    --publish=7474:7474 --publish=7687:7687 \
    --env NEO4J_AUTH=neo4j/neo4j@1234 \
    --volume=./data_neo4j:/data \
    --name neo4j-iteso \
    --network spark-cluster_default \
    neo4j:community-trixie
```

## Write GraphFrames into Neo4J

In [10]:
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

g.vertices.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .save()

print(f"{g.vertices.count()} verticess wrote in Neo4j")


g.edges.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("relationship", "FOLLOWS") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "match") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":User") \
  .option("relationship.target.save.mode", "match") \
  .option("relationship.target.node.keys", "dst:id") \
  .save()

print(f"{g.edges.count()} edges wrote in Neo4j")

3 verticess wrote in Neo4j
3 edges wrote in Neo4j


In [11]:
su.spark.stop()